## Code to fine-tune the LLM models

#### First by using torchtune to finetune llama
llama didn't work because requires licence approad so tried qwen

torchtune didn't work as it's very integrated with cuda. Tried MLX-LM

In [81]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [2]:
# !python -m torchtune._cli.tune ls

In [3]:
!tune ls

#need to install torchao separately

Skipping import of cpp extensions due to incompatible torch version 2.14.1 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
W1004 15:54:45.636000 11882 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
RECIPE                                   CONFIG                                  
full_finetune_single_device              llama2/7B_full_low_memory               
                                         code_llama2/7B_full_low_memory          
                                         llama3/8B_full_single_device            
                                         llama3_1/8B_full_single_device          
                                         llama3_2/1B_full_single_device          
                                         llama3_2/3B_full_single_device          
                                         mistral/7B_full_low_memory              
                      

Load dataset using huggingface

In [6]:
from datasets import load_dataset, Dataset

ds = load_dataset('keivalya/MedQuad-MedicalQnADataset', split="train")
filtered_ds = ds[:500]

README.md:   0%|          | 0.00/233 [00:00<?, ?B/s]

medDataset_processed.csv: reconstructing file:   0%|          |  0.00B / 22.5MB            

medDataset_processed.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/16407 [00:00<?, ? examples/s]

In [12]:
filtered_ds_2 = Dataset.from_dict(filtered_ds)

In [13]:
filtered_ds_2[0]

{'qtype': 'susceptibility',
 'Question': 'Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?',
 'Answer': 'LCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials from infected rodents.  Transmission may also occur when these materials are directly introduced into broken skin, the nose, the eyes, or the mouth, or presumably, via the bite of an infected rodent. Person-to-person transmission has not been reported, with the exception of vertical transmission from infected mother to fetus, and rarely, through organ transplantation.'}

In [20]:
# Intent database
from datasets import load_dataset, Dataset, load_from_disk

In [ ]:


ds = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset', split="train")
filtered_ds = Dataset.from_dict(ds[:500])


def create_intent_example(row):
    # Fill out the columns in the prompt
    row['intent_example'] =  f"Query: {row['instruction']}\nIntent: {row['intent']}"
    return row

# Call the ds method to apply our preprocessing function to all rows
processed_dataset = filtered_ds.map(create_intent_example)
# Print the intent_example in the first row of the processed data
print(processed_dataset[0]["intent_example"])

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Query: question about cancelling order {{Order Number}}
Intent: cancel_order


In [108]:
processed_dataset

Dataset({
    features: ['flags', 'instruction', 'category', 'intent', 'response', 'intent_example'],
    num_rows: 500
})

In [19]:
processed_dataset.save_to_disk("./datasets/customer_intent.dataset")

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
ds_processed = load_from_disk("./datasets/customer_intent.dataset")

In [106]:
ds_processed[0]

{'flags': 'B',
 'instruction': 'question about cancelling order {{Order Number}}',
 'category': 'ORDER',
 'intent': 'cancel_order',
 'response': "I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.",
 'intent_example': 'Query: question about cancelling order {{Order Number}}\nIntent: cancel_order'}

#### Generating configuration

In [ ]:
# config_dict = {
#     # Define the model
#     "dtype": "fp32",
#     "model":{"_component_":"torchtune.models.llama3_2.llama3_2_1b"},
#     # Define the batch size
#     "batch_size": 1,
#     "gradient_accumulation_steps": 4,
#     # Define the device type
#     "device": "mps",
#     "epochs": 2,
#     "max_steps_per_epoch": "null", 
#     "compile": False,
#     "resume_from_checkpoint": False,
#     "profiler": {
#         "_component_": "torchtune.training.setup_torch_profiler",
#         "enabled": False
#     },

#     "optimizer": {"_component_": "torch.optim.AdamW", "lr": 3e-05},
#     "dataset": {
#         "_component_": "torchtune.datasets.instruct_dataset",
#         "source": "arrow",
#         "data_files": "/Users/aakashsahu/Desktop/learning/open-weight-models/datasets/customer_intent.dataset",
#         "max_seq_len": 512,
#         "column_map": {
#             "input": "instruction",
#             "output": "intent"
#         }
#         },
#     "seed": 42,    
#     "output_dir": "/Users/aakashsahu/Desktop/learning/open-weight-models/models/finetune_results",
#     "checkpoint_dir": "/Users/aakashsahu/Desktop/learning/open-weight-models/models/fine-tuned/tmp/Llama-3.2-1B-Instruct",
#     "checkpoint_files": [
#     "/Users/aakashsahu/Desktop/learning/open-weight-models/models/fine-tuned/tmp/meta_model_0.pt"
#   ]
# }

In [67]:
# llama3_2/1B_full_single_device

In [ ]:
# config_dict = {
#     # Update the model
#     "model": { "_component_": "torchtune.models.llama3_2.llama3_2_3b" },
#     "batch_size": 8,
#     "device": "cuda",
#     "optimizer": {"_component_": "bitsandbytes.optim.PagedAdamW8bit", "lr": 3e-05},
#     "dataset": {"_component_": "custom_dataset"},
#     "output_dir": "/tmp/finetune_results"
# }

# import yaml
# # Save the updated configuration to a new YAML file
# with open("custom_fine_tune_recipe.yaml", "w") as yaml_file:
#     yaml.dump(config_dict, yaml_file)

In [14]:
# os.environ["hf_token"]

In [ ]:
!tune download Qwen/Qwen2.5-1.5B-Instruct --output-dir /Users/aakashsahu/Desktop/learning/open-weight-models/models/Qwen2.5-1.5B-Instruct --hf-token ""

Skipping import of cpp extensions due to incompatible torch version 2.14.1 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
W1004 20:27:00.538000 14370 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Ignoring files matching the following patterns: None
Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 10 files:   0%|                                | 0/10 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0%|       |  0.00B / 3.09GB            
Reconstructing (incomplete total...):   0%|       |  0.00B / 3.09GB            

Fetching 10 files:  10%|██▍                     | 1/10 [00:00<00:02,  3.01it/s]
Reconstructing (incomplete total...):   0%|       | 11.3kB / 3.09GB            
Reconstructing (incomplete total...):   0%|       | 11.3kB / 3.09GB            
Reconstructing (incomplete total...):   0%|       | 12.0kB / 3.

In [97]:
!tune run lora_finetune_single_device --config custom_fine_tune_recipe.yaml

Skipping import of cpp extensions due to incompatible torch version 2.14.1 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
W1004 20:51:06.576000 14787 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Running LoRAFinetuneRecipeSingleDevice with resolved config:

batch_size: 1
checkpointer:
  _component_: torchtune.training.FullModelHFCheckpointer
  checkpoint_dir: /Users/aakashsahu/Desktop/learning/open-weight-models/models/Qwen2.5-1.5B-Instruct
  checkpoint_files:
  - model.safetensors
  model_type: QWEN2
  output_dir: /Users/aakashsahu/Desktop/learning/open-weight-models/models/finetune_results
compile: false
dataset:
  _component_: torchtune.datasets.instruct_dataset
  column_map:
    input: instruction
    output: intent
  data_files: /Users/aakashsahu/Desktop/learning/open-weight-models/datasets/customer_intent.dataset
  max_seq_len: 512
  source: arrow
device: 

In [102]:
! python -c "import torch; torch.cpu.memory_stats = lambda: {}; torch.cpu.max_memory_allocated = lambda *args, **kwargs: 0; torch.cpu.max_memory_reserved = lambda *args, **kwargs: 0; import torchtune._cli.tune as tune; tune.main()" run lora_finetune_single_device --config custom_fine_tune_recipe.yaml

Skipping import of cpp extensions due to incompatible torch version 2.14.1 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
W1004 20:56:13.587000 14889 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Running LoRAFinetuneRecipeSingleDevice with resolved config:

batch_size: 1
checkpointer:
  _component_: torchtune.training.FullModelHFCheckpointer
  checkpoint_dir: /Users/aakashsahu/Desktop/learning/open-weight-models/models/Qwen2.5-1.5B-Instruct
  checkpoint_files:
  - model.safetensors
  model_type: QWEN2
  output_dir: /Users/aakashsahu/Desktop/learning/open-weight-models/models/finetune_results
compile: false
dataset:
  _component_: torchtune.datasets.instruct_dataset
  column_map:
    input: instruction
    output: intent
  data_files: /Users/aakashsahu/Desktop/learning/open-weight-models/datasets/customer_intent.dataset
  max_seq_len: 512
  source: arrow
device: 

In [ ]:
!tune cp lora_finetune_single_device my_custom_config.yaml
# this downloaded a python file

Skipping import of cpp extensions due to incompatible torch version 2.14.1 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
W1004 19:31:33.057000 12951 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Copied file to my_custom_config.py


In [35]:
!tune cp llama3_2/1B_lora_single_device my_mac_config.yaml

Skipping import of cpp extensions due to incompatible torch version 2.14.1 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
W1004 19:39:35.702000 13167 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Copied file to my_mac_config.yaml


## trying the mlx approach
Keep running into error with torchtune

In [109]:
# 2. Setup your brand new MLX target directory
mlx_data_dir = "/Users/aakashsahu/Desktop/learning/open-weight-models/datasets/mlx_customer_intent"
os.makedirs(mlx_data_dir, exist_ok=True)

# 3. Create a formatting function to structure your text for Qwen (ChatML format)
def format_for_mlx(row):
    return {
        "text": f"<|im_start|>user\n{row['instruction']}<|im_end|>\n<|im_start|>assistant\n{row['response']}<|im_end|>"
    }

formatted_dataset = processed_dataset.map(format_for_mlx, remove_columns=processed_dataset.column_names)
splits = formatted_dataset.train_test_split(test_size=0.05, seed=42)

splits["train"].to_json(os.path.join(mlx_data_dir, "train.jsonl"))
splits["test"].to_json(os.path.join(mlx_data_dir, "valid.jsonl"))

print(f"Export complete! Files stored inside: {mlx_data_dir}")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Export complete! Files stored inside: /Users/aakashsahu/Desktop/learning/open-weight-models/datasets/mlx_customer_intent


In [110]:
! mlx_lm.lora \
  --model /Users/aakashsahu/Desktop/learning/open-weight-models/models/Qwen2.5-1.5B-Instruct \
  --data /Users/aakashsahu/Desktop/learning/open-weight-models/datasets/mlx_customer_intent  \
  --train \
  --batch-size 1 \
  --iters 1000


Loading pretrained model
Loading datasets
Training
╭─ mlx_lm.lora ────────────────────────────────────────────────────────────────╮
│    model                                                                     │
│  /Users/aakashsahu/Desktop/learning/open-weight-models/models/Qwen2.5-1.5B-  │
│  Instruct                                                                    │
│    type            lora · 16 layers · rank 8                                 │
│    dataset                                                                   │
│  /Users/aakashsahu/Desktop/learning/open-weight-models/datasets/mlx_custome  │
│  r_intent                                                                    │
│    optimizer       adam · lr 1.0e-05                                         │
│    batch · iters   1 · 1,000                                                 │
│    max seq         2,048                                                     │
╰─────────────────────────────────────────────────────────

In [ ]:
## use fined tuned model for inference
!mlx_lm.generate \
  --model /Users/aakashsahu/Desktop/learning/open-weight-models/models/Qwen2.5-1.5B-Instruct \
  --adapter-path ./adapters \
  --prompt "question about cancelling order 55432" \
  --max-tokens 100

I've understood that you have a question about canceling order 55432. I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.
Prompt: 39 tokens, 89.027 tokens-per-sec
Generation: 50 tokens, 35.708 tokens-per-sec
Peak memory: 3.229 GB


In [114]:
# save as a standalone model
!mlx_lm.fuse \
  --model /Users/aakashsahu/Desktop/learning/open-weight-models/models/Qwen2.5-1.5B-Instruct \
  --adapter-path ./adapters \
  --save-path ./models/my-fused-qwen-customer-intent

Loading pretrained model


In [ ]:
!cp /Users/aakashsahu/Desktop/learning/open-weight-models/models/Qwen2.5-1.5B-Instruct/*.json ./models/my-fused-qwen-customer-intent/

In [121]:
!curl -L -O http://raw.githubusercontent.com/ggml-org/llama.cpp/refs/heads/master/convert_hf_to_gguf.py

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 13012  100 13012    0     0  39931      0 --:--:-- --:--:-- --:--:-- 39931


In [126]:
!git clone --depth 1 --filter=blob:none --sparse https://github.com/ggml-org/llama.cpp.git llama_converter
!cd llama_converter
!git sparse-checkout set convert_hf_to_gguf.py conversion gguf-py

Cloning into 'llama_converter'...
remote: Enumerating objects: 378, done.
remote: Counting objects: 100% (378/378), done.
remote: Compressing objects: 100% (323/323), done.
Receiving objects: 100% (378/378), 130.41 KiB | 3.03 MiB/s, done.
remote: Total 378 (delta 0), reused 303 (delta 0), pack-reused 0 (from 0)
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 31 (delta 0), reused 21 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 93.41 KiB | 601.00 KiB/s, done.


In [3]:
!python llama.cpp/convert_hf_to_gguf.py ./models/my-fused-qwen-customer-intent \
  --outfile ./models/customer_routing_f16.gguf \
  --outtype f16

INFO:hf-to-gguf:Loading model: my-fused-qwen-customer-intent
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.bfloat16 --> F16, shape = {1536, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.bfloat16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.bfloat16 --> F16, shape = {8960, 1536}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.bfloat16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.bfloat16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.bfloat16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.bfloat16 --> F32, shape = {256}
INFO:hf-to-gguf:blk.0.attn_k.weight,  

In [ ]:
# !brew install cmake
!cd llama.cpp
# Compile the llama.cpp project (optimized for Apple Silicon / Metal)
!cmake -B build
!cmake --build build --config Release --target llama-quantize

# RUN in command line instead

CMake Error: The source directory "/Users/aakashsahu/Desktop/learning/open-weight-models" does not appear to contain CMakeLists.txt.
Specify --help for usage, or press the help button on the CMake GUI.
Unknown argument -target
Usage: cmake --build <dir>             [options] [-- [native-options]]
       cmake --build --preset <preset> [options] [-- [native-options]]
Options:
  <dir>          = Project binary directory to be built.
  --preset <preset>, --preset=<preset>
                 = Specify a build preset.
  --presets-file <file>, --presets-file=<file>
                 = Specify the path to a presets file.
  --list-presets[=<type>]
                 = List available build presets.
  --parallel [<jobs>], -j [<jobs>]
                 = Build in parallel using the given number of jobs. 
                   If <jobs> is omitted the native build tool's 
                   default number is used.
                   The CMAKE_BUILD_PARALLEL_LEVEL environment variable
                   spe

In [8]:
!./llama.cpp/build/bin/llama-quantize ./models/customer_routing_f16.gguf ./models/customer_routing_q5.gguf Q5_K_M

ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library
ggml_metal_library_compile_all: compiled 'fa_aux' library in 0.193 sec
ggml_metal_library_compile_all: compiled 'fa_f16' library in 7.015 sec
ggml_metal_library_compile_all: compiled 'fa_f32' library in 0.646 sec
ggml_metal_library_compile_all: compiled 'fa_q4_0' library in 3.495 sec
ggml_metal_library_compile_all: compiled 'fa_q4_1' library in 1.455 sec
ggml_metal_library_compile_all: compiled 'fa_q5_0' library in 9.399 sec
ggml_metal_library_compile_all: compiled 'fa_q5_1' library in 0.800 sec
ggml_metal_library_compile_all: compiled 'fa_q8_0' library in 9.077 sec
ggml_metal_library_compile_all: compiled 'fa_vec_f16' library in 5.313 sec
ggml_metal_library_compile_all: compiled 'fa_vec_f32' library in 0.921 sec
ggml_metal_library_compile_all: compiled 'fa_vec_q4_0' library in 5.327 sec
ggml_metal_library_compile_all: compiled 'fa_vec_q4_1' library in 3.374 

In [10]:
from llama_cpp import Llama
from pprint import pprint

model_path = "./models/customer_routing_q5.gguf"
llm = Llama(model_path)
# output = llm("cancel my order", max_tokens=1000)

llama_model_loader: loaded meta data with 31 key-value pairs and 338 tensors from ./models/customer_routing_q5.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 0.700000
llama_model_loader: - kv   5:            general.sampling.penalty_repeat f32              = 1.100000
llama_model_loader: - kv   6:                               general.name str              = My Fused Qwen Customer Intent
llama_model_loader: - kv   7:              

In [12]:
prompt = "cancel my order"

conv = [
    {
        "role": "user",
        "content": prompt
    }
]

result = llm.create_chat_completion(messages=conv, max_tokens=20)
print(result)

Llama.generate: 24 prefix-match hit, remaining 8 prompt tokens to eval
llama_perf_context_print:        load time =     306.17 ms
llama_perf_context_print: prompt eval time =      62.59 ms /     8 tokens (    7.82 ms per token,   127.81 tokens per second)
llama_perf_context_print:        eval time =     230.10 ms /    19 runs   (   12.11 ms per token,    82.57 tokens per second)
llama_perf_context_print:       total time =     295.60 ms /    27 tokens
llama_perf_context_print:    graphs reused =         18


{'id': 'chatcmpl-238aa310-e10d-404f-b25f-188689a70c73', 'object': 'chat.completion', 'created': 1791166544, 'model': './models/customer_routing_q5.gguf', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': "I've understood that you would like to cancel your order. I apologize for any inconvenience caused. To"}, 'logprobs': None, 'finish_reason': 'length'}], 'usage': {'prompt_tokens': 32, 'completion_tokens': 20, 'total_tokens': 52}}


In [13]:
print(result["choices"][0]["message"]["content"])

I've understood that you would like to cancel your order. I apologize for any inconvenience caused. To
